# 第三章 Notebook 1：MIDI 渲染基础

同一份 MIDI 文件通过不同音色和渲染参数，可以产生不同的声音。
本 Notebook 包含以下内容：

1. **MIDI 渲染链路**：MIDI 事件 → 软件合成器 → SoundFont → 音频文件
2. **音色切换**：用 Program Change 改变乐器音色
3. **渲染对比**：同一旋律在不同音色下的听感差异
4. **多声部渲染**：BWV 854 赋格的音色与织体关系
5. **渲染 ≠ 真实演奏**：符号与声音之间的信息鸿沟
6. **波形初探**：从符号到声音，衔接第 5 章音频表示

> 加法、减法与相位调制的代码实验见 `02_synthesis_primer.ipynb`。

数据：
- `CODE/datasets/melodies/红河谷.midi`
- `CODE/datasets/lmd_clean_midi/Bach Johann Sebastian/Bach Prelude and Fugue in E major BWV 854 Fugue.mid`

依赖：`pretty_midi`、`soundfile`、`PyYAML`、`fluidsynth`（命令行程序）和 `IPython.display.Audio`

---

**前置知识检查**

本 Notebook 假设你已了解以下内容。如果不熟悉，建议先浏览相关文档：
- Python 基础语法（函数、列表、字典）
- `numpy` 数组的基本操作
- 命令行工具的基本使用（后续代码将调用 `fluidsynth` CLI）
- MIDI 文件的基本概念（音高、力度、音色编号）

> 若 `fluidsynth` 未安装，可参考 FluidSynth 官方文档进行安装。


In [ ]:
import hashlib
import os
import re
import subprocess
import tempfile
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pretty_midi
import soundfile as sf
from IPython.display import Audio, display, Markdown

# 中文字体
plt.rcParams["font.sans-serif"] = [
    "PingFang SC", "Hiragino Sans GB", "Microsoft YaHei",
    "SimHei", "Arial Unicode MS", "Noto Sans CJK SC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

# 强制白底
for _k in ("figure.facecolor", "axes.facecolor"):
    plt.rcParams[_k] = "white"
for _k in ("axes.edgecolor", "axes.labelcolor", "xtick.color", "ytick.color", "text.color"):
    plt.rcParams[_k] = "black"

# 从当前目录向上定位项目根目录。
_p = os.getcwd()
while not os.path.exists(os.path.join(_p, "CODE", "datasets")):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError("未找到包含 CODE/datasets 的项目根目录")
    _p = _parent
BASE_DIR = _p
NOTEBOOK_DIR = os.path.join(BASE_DIR, "CODE", "chapter03")

HONGHE_MIDI = os.path.join(BASE_DIR, "CODE", "datasets", "melodies", "红河谷.midi")
BWV854_MIDI = os.path.join(
    BASE_DIR, "CODE", "datasets", "lmd_clean_midi", "Bach Johann Sebastian",
    "Bach Prelude and Fugue in E major BWV 854 Fugue.mid"
)
SOUNDFONT = os.path.join(BASE_DIR, "CODE", "datasets", "soundfonts", "TimGM6mb.sf2")

# 项目未附带 SoundFont 时，使用当前 pretty_midi 安装包中的 TimGM6mb。
if not os.path.exists(SOUNDFONT):
    import pretty_midi as _pm
    SOUNDFONT = os.path.join(os.path.dirname(_pm.__file__), "TimGM6mb.sf2")

if not os.path.exists(SOUNDFONT):
    raise FileNotFoundError(f"未找到 SoundFont: {SOUNDFONT}")

EXPECTED_SOUNDFONT_SHA256 = "82475b91a76de15cb28a104707d3247ba932e228bada3f47bba63c6b31aaf7a1"
with open(SOUNDFONT, "rb") as f:
    soundfont_sha256 = hashlib.file_digest(f, "sha256").hexdigest()
if soundfont_sha256 != EXPECTED_SOUNDFONT_SHA256:
    raise ValueError("TimGM6mb.sf2 的 SHA-256 与本 Notebook 验证版本不一致")

try:
    fluidsynth_info = subprocess.run(
        ["fluidsynth", "--version"], capture_output=True, text=True, check=True
    )
except FileNotFoundError as exc:
    raise RuntimeError("未找到 fluidsynth 命令行程序") from exc
fluidsynth_version_lines = (fluidsynth_info.stdout + fluidsynth_info.stderr).splitlines()
if not fluidsynth_version_lines:
    raise RuntimeError("fluidsynth --version 未返回版本信息")
fluidsynth_version = fluidsynth_version_lines[0].strip()
# 各平台 --version 输出的行序不一（版本行未必是首行），选取含版本号的行用于展示。
for _line in fluidsynth_version_lines:
    if re.search(r"version", _line, re.IGNORECASE) and re.search(r"\d+\.\d+", _line):
        fluidsynth_version = _line.strip()
        break

FIGURES_DIR = os.path.join(NOTEBOOK_DIR, "output_figures")
AUDIO_DIR = os.path.join(NOTEBOOK_DIR, "output_audio")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

def project_path(path):
    return os.path.relpath(path, BASE_DIR)


def notebook_path(path):
    return os.path.relpath(path, NOTEBOOK_DIR)


print("项目根目录：.")
print(f"SoundFont：{project_path(SOUNDFONT) if SOUNDFONT.startswith(BASE_DIR) else os.path.basename(SOUNDFONT)}")
print(f"SoundFont SHA-256：{soundfont_sha256}")
print(f"pretty_midi：{pretty_midi.__version__}；{fluidsynth_version}")
print(f"红河谷 MIDI：{project_path(HONGHE_MIDI)}")
print(f"BWV 854 MIDI：{project_path(BWV854_MIDI)}")


## 1. MIDI 渲染链路

MIDI 文件保存音符、控制器和速度（tempo）等事件，不保存音频波形。
在本 Notebook 的配置中，MIDI 通过以下链路转换为音频：

```
MIDI 事件 → 软件合成器（FluidSynth）→ SoundFont 音色库 → 音频文件（WAV）
```

- **软件合成器**：读取 MIDI 事件，根据音色库生成音频信号
- **SoundFont**：通过预设、乐器区域、采样、生成器和调制器参数定义音色
- **Program Change**：选择音色程序；General MIDI（GM）规定了 128 个旋律乐器程序

下面用 FluidSynth + TimGM6mb SoundFont 渲染《红河谷》。

In [ ]:
def render_midi(midi_path, wav_path, soundfont=SOUNDFONT, sample_rate=44100):
    """用 FluidSynth CLI 将 MIDI 渲染为 WAV。"""
    cmd = [
        "fluidsynth",
        "-F", wav_path,
        "-r", str(sample_rate),
        "-g", "2.0",
        "-ni",
        soundfont,
        midi_path,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"FluidSynth 渲染失败:\n{result.stderr}")
    return wav_path


def change_program(midi_path, program, output_path=None):
    """修改非打击乐轨的 Program Change，并返回路径和元事件布局警告状态。"""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", RuntimeWarning)
        _pm = pretty_midi.PrettyMIDI(midi_path)
    known_prefix = "Tempo, Key or Time signature change events found on non-zero tracks"
    unexpected = [w for w in caught if not str(w.message).startswith(known_prefix)]
    if unexpected:
        raise RuntimeError(f"读取 MIDI 时出现未处理警告：{unexpected[0].message}")
    metadata_layout_warning = any(str(w.message).startswith(known_prefix) for w in caught)
    for instrument in _pm.instruments:
        if not instrument.is_drum:
            instrument.program = program
    if output_path is None:
        fd, output_path = tempfile.mkstemp(suffix=".mid")
        os.close(fd)
    _pm.write(output_path)
    return output_path, metadata_layout_warning


# 渲染红河谷 — 钢琴 (GM 0)
piano_wav = os.path.join(AUDIO_DIR, "honghe_piano.wav")
render_midi(HONGHE_MIDI, piano_wav)

audio_data, sr = sf.read(piano_wav)
if audio_data.ndim == 2:
    audio_mono = audio_data.mean(axis=1)
else:
    audio_mono = audio_data

print(f"渲染完成：{project_path(piano_wav)}")
print(f"采样率：{sr} Hz；时长：{len(audio_mono)/sr:.2f} 秒")
print(f"波形范围：[{audio_mono.min():.4f}, {audio_mono.max():.4f}]")

In [ ]:
display(Markdown("**红河谷 — 钢琴 (GM 0)**"))
Audio(url=notebook_path(piano_wav))

## 2. 音色切换：Program Change

General MIDI 定义了 128 个旋律乐器程序。以下代码使用从 0 开始的程序编号；部分软件界面显示为 1–128。通过修改 Program Change，
同一段旋律可以使用不同的预设发声。

| GM 编号 | 音色名称 | 类别 |
|---------|---------|------|
| 0 | Acoustic Grand Piano | 钢琴 |
| 6 | Harpsichord | 键盘 |
| 19 | Church Organ | 管风琴 |
| 40 | Violin | 弦乐 |
| 73 | Flute | 木管 |

> 表中只列出本章使用的程序。GM 程序表也包含部分亚洲乐器名称，但没有以二胡或古琴命名的标准程序；这类音色通常需要专用音色库和相应映射。

In [ ]:
TIMBRES = {
    "钢琴":     {"program": 0,  "filename": "honghe_piano.wav"},
    "羽管键琴": {"program": 6,  "filename": "honghe_harpsichord.wav"},
    "长笛":     {"program": 73, "filename": "honghe_flute.wav"},
    "小提琴":   {"program": 40, "filename": "honghe_violin.wav"},
    "管风琴":   {"program": 19, "filename": "honghe_organ.wav"},
}

metadata_layout_warning = False
for name, cfg in TIMBRES.items():
    out_wav = os.path.join(AUDIO_DIR, cfg["filename"])
    tmp_mid, warned = change_program(HONGHE_MIDI, cfg["program"])
    metadata_layout_warning = metadata_layout_warning or warned
    try:
        render_midi(tmp_mid, out_wav)
    finally:
        os.remove(tmp_mid)
    print(f"  {name}（GM {cfg['program']}）：{project_path(out_wav)}")

print("\n全部渲染完成")
if metadata_layout_warning:
    print("说明：源文件把调号元事件放在非零轨；速度元事件位于轨 0。")
    print("本例只修改程序编号，并保留 pretty_midi 解析的 60 BPM 时间轴。")

In [ ]:
for name, cfg in TIMBRES.items():
    out_wav = os.path.join(AUDIO_DIR, cfg["filename"])
    display(Markdown(f"**红河谷 — {name} (GM {cfg['program']})**"))
    display(Audio(url=notebook_path(out_wav)))

## 3. 波形对比：同一旋律，不同声音

以下五次渲染保留原文件中的音符和控制器事件，只改变程序编号，并采用相同的 FluidSynth 增益、采样率和 TimGM6mb.sf2 文件。当前五个预设的波形存在差异；试听比较同样只适用于这些渲染条件。图中将双声道输出取均值，并统一五幅图的纵轴范围。

### 《红河谷》五种 GM 程序的渲染波形

In [ ]:
fig, axes = plt.subplots(len(TIMBRES), 1, figsize=(12, 2.2 * len(TIMBRES)), sharex=True)

# 先扫描所有音色的最大振幅，统一 y 轴范围
global_max = 0
for cfg in TIMBRES.values():
    d, _ = sf.read(os.path.join(AUDIO_DIR, cfg["filename"]))
    if d.ndim == 2:
        d = d.mean(axis=1)
    global_max = max(global_max, np.abs(d).max())
y_lim = global_max * 1.1

for ax, (name, cfg) in zip(axes, TIMBRES.items()):
    out_wav = os.path.join(AUDIO_DIR, cfg["filename"])
    data, sr = sf.read(out_wav)
    if data.ndim == 2:
        mono = data.mean(axis=1)
    else:
        mono = data
    t = np.arange(len(mono)) / sr
    ax.plot(t, mono, color="#2c3e50", linewidth=0.3)
    ax.set_ylabel(name, fontsize=10)
    ax.set_ylim(-y_lim, y_lim)
    ax.grid(axis="x", linestyle=":", alpha=0.3)

axes[-1].set_xlabel("时间（秒）")

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, "fig_waveform_compare.png")
fig.savefig(out_path, dpi=600, bbox_inches="tight")
plt.show()
print(f"已保存：{project_path(out_path)}")

## 4. 多声部渲染：BWV 854 赋格

下面截取 BWV 854 赋格第 1–17 小节，并分别用钢琴、羽管键琴和管风琴程序渲染。片段边界依据配套 MusicXML 与 MIDI 的起音对齐结果确定；播放时间保留 MIDI 文件的 120 BPM，不按 MusicXML 中的 175 BPM 标记重缩放。

TimGM6mb.sf2 的三个预设在衰减、持续特性和频谱上不同，同一组音符与踏板事件因而产生不同的重叠关系和声部融合程度。相关听觉比较只适用于本次渲染条件。

In [ ]:
import yaml

SIDECAR_PATH = os.path.join(
    BASE_DIR, "CODE", "datasets", "lmd_clean_midi", "Bach Johann Sebastian",
    "BWV854_fugue_sidecar.yaml"
)
with open(SIDECAR_PATH, "r", encoding="utf-8") as f:
    sidecar = yaml.safe_load(f)

# 读取配套 MusicXML--MIDI 对齐结果，并保留 SMF 的播放时间轴。
bwv_pm = pretty_midi.PrettyMIDI(BWV854_MIDI)
tempo_changes = bwv_pm.get_tempo_changes()
midi_tempo = float(tempo_changes[1][0]) if len(tempo_changes[1]) > 0 else 120.0
smf_observed = sidecar["smf_observed"]
musicxml_observed = sidecar["musicxml_observed"]
alignment = sidecar["alignment"]
assert np.isclose(midi_tempo, float(smf_observed["tempo_meta_bpm"]))

all_notes = []
for inst in bwv_pm.instruments:
    if not inst.is_drum:
        all_notes.extend(inst.notes)
all_notes.sort(key=lambda n: n.start)

measure_starts = {int(item["measure"]): float(item["start"]) for item in alignment["measure_starts_seconds"]}
excerpt_start_measure = int(alignment["excerpt_start_measure"])
excerpt_end_measure = int(alignment["excerpt_end_measure"])
excerpt_measures = excerpt_end_measure - excerpt_start_measure + 1
excerpt_start = measure_starts[excerpt_start_measure]
excerpt_end = measure_starts[excerpt_end_measure + 1]
excerpt_duration = excerpt_end - excerpt_start
boundary_tolerance = 1e-6  # sidecar 秒值保留 10 位小数，容纳浮点舍入误差
excerpt_notes = [
    n for n in all_notes
    if excerpt_start - boundary_tolerance <= n.start < excerpt_end - boundary_tolerance
]
assert len(all_notes) == int(smf_observed["note_count"])
assert len(excerpt_notes) == int(alignment["excerpt_note_on_count"])
source_controls = [
    cc for inst in bwv_pm.instruments if not inst.is_drum
    for cc in inst.control_changes if cc.number in (7, 64)
]
expected_controller_counts = {}
for number in (7, 64):
    has_prior = any(cc.number == number and cc.time <= excerpt_start for cc in source_controls)
    inside_count = sum(
        cc.number == number and excerpt_start < cc.time < excerpt_end - boundary_tolerance
        for cc in source_controls
    )
    expected_controller_counts[number] = int(has_prior) + inside_count
expected_controller_counts[64] += 1  # 片段终点显式复位延音踏板


def write_excerpt_midi(program, output_path):
    """按对齐边界截取起音，保留 CC7/CC64，并写入指定音色。"""
    new_pm = pretty_midi.PrettyMIDI(initial_tempo=midi_tempo)
    meter_num, meter_den = map(int, musicxml_observed["notated_meter"].split("/"))
    new_pm.time_signature_changes.append(pretty_midi.TimeSignature(meter_num, meter_den, 0.0))
    new_inst = pretty_midi.Instrument(program=program, name="BWV 854 mm. 1-17")
    for n in excerpt_notes:
        start = n.start - excerpt_start
        end = min(n.end, excerpt_end) - excerpt_start
        if end <= start:
            continue
        note = pretty_midi.Note(
            velocity=n.velocity,
            pitch=n.pitch,
            start=start,
            end=end,
        )
        new_inst.notes.append(note)

    for number in (7, 64):
        prior = [cc for cc in source_controls if cc.number == number and cc.time <= excerpt_start]
        if prior:
            last = max(prior, key=lambda cc: cc.time)
            new_inst.control_changes.append(pretty_midi.ControlChange(number, last.value, 0.0))
        for cc in source_controls:
            if cc.number == number and excerpt_start < cc.time < excerpt_end - boundary_tolerance:
                new_inst.control_changes.append(
                    pretty_midi.ControlChange(number, cc.value, cc.time - excerpt_start)
                )
    new_inst.control_changes.append(pretty_midi.ControlChange(64, 0, excerpt_duration))
    new_inst.control_changes.sort(key=lambda cc: (cc.time, cc.number))
    new_pm.instruments.append(new_inst)
    new_pm.write(output_path)
    return output_path


BWV_TIMBRES = {
    "钢琴":     {"program": 0,  "filename": "bwv854_piano.wav"},
    "羽管键琴": {"program": 6,  "filename": "bwv854_harpsichord.wav"},
    "管风琴":   {"program": 19, "filename": "bwv854_organ.wav"},
}

for name, cfg in BWV_TIMBRES.items():
    out_wav = os.path.join(AUDIO_DIR, cfg["filename"])
    fd, tmp_mid = tempfile.mkstemp(suffix=".mid")
    os.close(fd)
    try:
        write_excerpt_midi(cfg["program"], tmp_mid)
        check_pm = pretty_midi.PrettyMIDI(tmp_mid)
        assert sum(len(inst.notes) for inst in check_pm.instruments) == len(excerpt_notes)
        actual_controller_counts = {
            number: sum(
                cc.number == number for inst in check_pm.instruments for cc in inst.control_changes
            )
            for number in (7, 64)
        }
        assert actual_controller_counts == expected_controller_counts
        render_midi(tmp_mid, out_wav)
    finally:
        os.remove(tmp_mid)
    print(f"  BWV 854 — {name}（GM {cfg['program']}）：{project_path(out_wav)}")

print()
print(f"SMF 速度元事件（用于播放）：{midi_tempo:.0f} BPM")
print(f"MusicXML 速度标记（仅报告）：{float(musicxml_observed['tempo_mark_bpm']):.0f} BPM")
print(f"MusicXML 记谱拍号：{musicxml_observed['notated_meter']}")
print(f"截取范围：第 {excerpt_start_measure}–{excerpt_end_measure} 小节（{excerpt_measures} 小节）")
print(f"对齐时间边界：{excerpt_start:.6f}–{excerpt_end:.6f} 秒")
print(f"片段起音数：{len(excerpt_notes)} / {len(all_notes)}")
print(
    "片段控制器事件数："
    f"CC7={expected_controller_counts[7]}；"
    f"CC64={expected_controller_counts[64]}（含片段终点复位）"
)


In [ ]:
for name, cfg in BWV_TIMBRES.items():
    out_wav = os.path.join(AUDIO_DIR, cfg["filename"])
    display(Markdown(f"**BWV 854 赋格 — {name} (GM {cfg['program']})**"))
    display(Audio(url=notebook_path(out_wav)))

## 5. 渲染不等于真实演奏

MIDI 能编码音符、力度、微时值、踏板、弯音、触后和其他控制器，但具体文件、协议版本、音源与映射决定了最终可用的信息。下表比较本 Notebook 的基础 MIDI 1.0 + TimGM6mb + FluidSynth 渲染与声学演奏，而不是概括所有 MIDI 系统：

| 维度 | 本 Notebook 的基础渲染 | 声学演奏中可出现的信息 |
|------|----------|----------|
| 时值 | 保留文件中已有的起止时间；本例未加入演奏性微调 | rubato、声部间微时差等 |
| 动态 | Note On 力度和 CC7；具体响应由音源决定 | 触键、气息或弓压的连续变化 |
| 音色 | TimGM6mb 的所选预设与区域映射 | 随演奏法和发声状态连续变化 |
| 空间 | FluidSynth 当前声道与效果设置 | 厅堂响应、乐器位置和收音方式 |
| 演奏法 | BWV 854 片段保留音符、CC7 和 CC64 | 滑音、揉弦、吟猱、气口、轮指等 |

弯音、控制器、扩展协议或专用采样映射可以承载二胡滑音和揉弦、古琴吟猱与绰注、笛子气声与花舌等演奏信息，但只有在音源建立相应映射时才会影响输出。本例的 GM 预设与事件集合没有提供这些映射，因此未显式表示这些演奏细节。

## 6. 波形初探：通往音频表示

渲染产生的音频由离散时间采样值组成；采样值表示数字音频信号的瞬时幅度。下面将《红河谷》钢琴渲染的双声道取均值，再观察一段波形。

### 《红河谷》钢琴程序的全局与局部波形

In [ ]:
piano_data, sr = sf.read(os.path.join(AUDIO_DIR, "honghe_piano.wav"))
if piano_data.ndim == 2:
    piano_mono = piano_data.mean(axis=1)
else:
    piano_mono = piano_data

# 放大前 0.5 秒的第一个音符区域
start_sample = int(0.05 * sr)
end_sample = int(0.55 * sr)
segment = piano_mono[start_sample:end_sample]
t = np.arange(len(segment)) / sr + 0.05

fig, axes = plt.subplots(2, 1, figsize=(12, 5))

# 全局波形
ax = axes[0]
t_full = np.arange(len(piano_mono)) / sr
ax.plot(t_full, piano_mono, color="#2c3e50", linewidth=0.3)
ax.axvspan(0.05, 0.55, alpha=0.15, color="orange")
ax.set_xlabel("时间（秒）")
ax.set_ylabel("振幅")
ax.set_xlim(0, len(piano_mono) / sr)

# 放大区域
ax = axes[1]
ax.plot(t, segment, color="#e74c3c", linewidth=0.5)
ax.set_xlabel("时间（秒）")
ax.set_ylabel("振幅")

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, "fig_waveform_zoom.png")
fig.savefig(out_path, dpi=600, bbox_inches="tight")
plt.show()
print(f"已保存：{project_path(out_path)}")
print()
print("时域波形显示信号幅度随时间的变化。")
print("仅凭短时波形通常难以直接判断音高和音色；第 5 章将进一步讨论频谱表示。")

## 小结

| 要点 | 说明 |
|------|------|
| MIDI 不是音频 | MIDI 保存事件；本 Notebook 的声音由 FluidSynth 和 SoundFont 生成 |
| 采样音源渲染 | FluidSynth 根据 SoundFont 的采样、区域与播放参数生成声音 |
| Program Change | 在当前 GM 音源中，改变程序编号可切换预设音色 |
| 同一符号，不同声音 | 音源、预设与渲染参数共同影响输出 |
| 织体与音色 | 当前三个预设对同一事件产生不同的重叠与融合效果 |
| 信息边界 | 渲染只能利用文件、协议、映射与音源实际提供的信息 |

当研究对象包括音色、空间或未编码的演奏细节时，需要直接分析音频信号；第 5 章将介绍相应表示。

> 采样音源渲染与加法、减法、相位调制的对照实验见 `02_synthesis_primer.ipynb`。

| 输出文件 | 内容 |
|----------|------|
| `output_audio/honghe_piano.wav` | 红河谷 — 钢琴音色 |
| `output_audio/honghe_harpsichord.wav` | 红河谷 — 羽管键琴音色 |
| `output_audio/honghe_flute.wav` | 红河谷 — 长笛音色 |
| `output_audio/honghe_violin.wav` | 红河谷 — 小提琴音色 |
| `output_audio/honghe_organ.wav` | 红河谷 — 管风琴音色 |
| `output_audio/bwv854_piano.wav` | BWV 854 — 钢琴音色 |
| `output_audio/bwv854_harpsichord.wav` | BWV 854 — 羽管键琴音色 |
| `output_audio/bwv854_organ.wav` | BWV 854 — 管风琴音色 |
| `output_figures/fig_waveform_compare.png` | 波形对比图 |
| `output_figures/fig_waveform_zoom.png` | 波形放大图 |
